# Basic Optics and Nonlinear Driving Terms

This notebook demonstrates the main optics workflow in `simplestoragering`: build a storage ring magnetic focusing structure, solve the linear optics, plot optical functions, inspect the RF bucket, calculate nonlinear driving terms, evaluate amplitude-dependent tune shifts, study off-momentum optics, and adjust chromaticities and tunes.

## 1. Prepare Imports

The path setup below lets the notebook run from either the project root or `docs/notebooks/`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    p for p in (cwd, *cwd.parents)
    if (p / "simplestoragering").exists() and (p / "examples").exists()
)

sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "examples"))

print(f"Project root: {project_root}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import PyNAFF as pnf

import simplestoragering as ssr
from simplestoragering.objectives import quantify_rdt_fluctuation
from example_lattice import generate_ring

plt.rcParams["figure.figsize"] = (8, 4)

## 2. Generate the Ring

`generate_ring()` is defined in `examples/example_lattice.py`. Keeping the magnetic focusing structure in a plain Python file makes it reusable from scripts, tests, and notebooks.

In [ ]:
ring = generate_ring()
ring.linear_optics()

print(ring)

## 3. Plot Linear Optical Functions

The package provides convenience plotting helpers for optics and magnet layout.

In [ ]:
ssr.plot_lattice(ring, ["betax", "betay"])
plt.show()

For smoother curves, slice long elements before extracting element-by-element columns.

In [ ]:
sliced = ring.slice_elements(
    drift_length=0.1,
    bend_length=0.01,
    quad_length=0.01,
    sext_length=0.01,
    oct_length=0.01,
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ssr.get_col(sliced, "s"), ssr.get_col(sliced, "etax"))
ax.set_xlabel("s [m]")
ax.set_ylabel(r"$\eta_x$ [m]")
ax.grid(True)
ssr.plot_layout_in_ax(ring.elements, ax.twinx())
plt.show()

## 4. Plot the RF Bucket

The RF bucket plot uses the ring circumference to set the RF frequency.

In [ ]:
rf_params = {
    "voltage": 0.92,
    "frequency": 560 * 299792458 / ring.length,
    "harmonic_number": 560,
}

ssr.plot_rf_bucket(ring, rf_params)
plt.show()

## 5. Track Chromaticity

`ring.xi_x` and `ring.xi_y` are obtained from the linear optics calculation. Tracking-based chromaticities can be used as a cross-check.

In [ ]:
tracked_chromaticity = ring.track_chromaticity(order=4, delta=1e-3, verbose=True)
tracked_chromaticity

## 7. Nonlinear Driving Terms

The amplitude-dependent tune shift (ADTS) terms can be estimated directly from the driving terms.

In [ ]:
rdts = ring.driving_terms(verbose=True)

f_rms = quantify_rdt_fluctuation(rdts, w=1 / 200)
f3_rms, f4_rms = quantify_rdt_fluctuation(rdts, w=0)
adts = rdts.adts()
adts
print(f"f_rms : {f_rms:.2f}")
print(f"f3_rms: {f3_rms:.2f}")
print(f"f4_rms: {f4_rms:.2f}")

In [ ]:
data = ssr.plot_RDTs_along_ring(ring, RDT_type="f")

## 8. Off-Momentum Optics

Off-momentum optics modifies the closed orbit and optical functions. Re-run `linear_optics()` after this section if you want to restore the on-momentum state.

In [ ]:
delta = 0.03

try:
    ring.off_momentum_optics(delta=delta)
    off_momentum_rdts = ring.driving_terms(verbose=False)
    off_f_rms = quantify_rdt_fluctuation(off_momentum_rdts, w=1 / 200)
    off_f3_rms, off_f4_rms = quantify_rdt_fluctuation(off_momentum_rdts, w=0)
    print(f"delta = {delta:+.1%}")
    print(f"nux   = {ring.nux:.4f}")
    print(f"nuy   = {ring.nuy:.4f}")
    print(f"betax at start = {ring.elements[0].betax:.4f}")
    print(f"f_rms = {off_f_rms:.2f}, f3_rms = {off_f3_rms:.2f}, f4_rms = {off_f4_rms:.2f}")
except ssr.Unstable:
    print(f"The ring is unstable at delta = {delta:+.1%}.")

Restore the on-momentum linear optics before making further corrections.

In [ ]:
ring.linear_optics()
print(f"Restored on-momentum tunes: nux = {ring.nux:.4f}, nuy = {ring.nuy:.4f}")

## 9. Adjust Chromaticities and Tunes

The following cells demonstrate simple matching utilities. They modify magnet strengths in the current `ring` object.

In [ ]:
k2_list = ssr.chromaticity_correction(
    ring,
    sextupole_name_list=["SF3", "SD3"],
    target=[3, 3],
)
ring.linear_optics()

print(f"new SF3 k2 = {k2_list[0]:.6f}")
print(f"new SD3 k2 = {k2_list[1]:.6f}")
print(f"xi_x = {ring.xi_x:.2f}, xi_y = {ring.xi_y:.2f}")

In [ ]:
k1_list = ssr.adjust_tunes(
    ring,
    quadrupole_name_list=["Q1", "Q2"],
    target=[43.4, 16.4],
    iterations=5,
)

print(f"new Q1 k1 = {k1_list[0]:.6f}")
print(f"new Q2 k1 = {k1_list[1]:.6f}")
print(f"nux = {ring.nux:.4f}, nuy = {ring.nuy:.4f}")